# Python Multiprocessing — True Parallelism, GIL Bypass & Process Pools

> **Topic:** Multiprocessing & Parallel Computing | **Folder:** Concurrency

Python's **`multiprocessing`** module enables **true multi-core parallel execution** by spawning separate
OS processes, each with its own Python interpreter and memory space.

By running multiple processes, `multiprocessing` completely **bypasses the Global Interpreter Lock (GIL)**,
making it the standard choice for **CPU-bound algorithmic workloads**.

---

## Table of Contents
1. [Concurrency vs. Parallelism & The GIL](#1.-Concurrency-vs.-Parallelism-&-The-GIL)
2. [The `multiprocessing.Process` API](#2.-The-`multiprocessing.Process`-API)
3. [Process Pools (`multiprocessing.Pool` & `ProcessPoolExecutor`)](#3.-Process-Pools-(multiprocessing.Pool-&-ProcessPoolExecutor))
4. [Inter-Process Communication (IPC: `Queue` & `Pipe`)](#4.-Inter-Process-Communication-(IPC:-Queue-&-Pipe))
5. [Shared Memory & Managers (`Value`, `Array`, `Manager`)](#5.-Shared-Memory-&-Managers-(Value,-Array,-Manager))
6. [Process Synchronization Primitives (`Lock`, `Semaphore`, `Event`)](#6.-Process-Synchronization-Primitives-(Lock,-Semaphore,-Event))
7. [Process Start Methods (`spawn`, `fork`, `forkserver`)](#7.-Process-Start-Methods-(spawn,-fork,-forkserver))
8. [Empirical Benchmark: Sequential vs. Threads vs. Processes](#8.-Empirical-Benchmark:-Sequential-vs.-Threads-vs.-Processes)
9. [Quick Reference Card](#9.-Quick-Reference-Card)


---
## 1. Concurrency vs. Parallelism & The GIL

| Paradigm | Mechanism | Best For | GIL Impact |
|----------|-----------|----------|------------|
| **Multithreading (`threading`)** | Multiple threads in 1 process | I/O-bound (network, disk) | **Blocked by GIL** (only 1 thread executes Python bytecode at a time) |
| **Multiprocessing (`multiprocessing`)** | Multiple distinct OS processes | CPU-bound (heavy math, data analysis) | **Bypasses GIL** (each process has its own GIL) |
| **Async I/O (`asyncio`)** | Single-threaded event loop | Network I/O (Web APIs, WebSockets) | Non-blocking single thread |


In [ ]:
# Checking CPU core count
import os
import multiprocessing as mp

cpu_count = os.cpu_count()
print(f"Available CPU Cores (Logical Processors): {cpu_count}")
print(f"Multiprocessing start method          : {mp.get_start_method()}")


---
## 2. The `multiprocessing.Process` API

Creating a process requires passing a target function and tuple arguments `args`.
Always guard process instantiation inside `if __name__ == '__main__':` to prevent recursive process spawning!


In [ ]:
import multiprocessing as mp
import time

def worker_task(name, duration):
    print(f"  [Process-{name}] Starting execution (PID: {os.getpid()})...")
    time.sleep(duration)
    print(f"  [Process-{name}] Execution complete.")

if __name__ == '__main__' or True:  # Jupyter safe demonstration
    p1 = mp.Process(target=worker_task, args=("A", 0.2))
    p2 = mp.Process(target=worker_task, args=("B", 0.3))
    
    p1.start()  # Launch process 1
    p2.start()  # Launch process 2
    
    p1.join()   # Wait for p1 to finish
    p2.join()   # Wait for p2 to finish
    print("All processes joined successfully.")


---
## 3. Process Pools (`multiprocessing.Pool` & `ProcessPoolExecutor`)

Process Pools manage a fixed pool of worker processes to distribute tasks parallelly across CPU cores.

| Method | Execution Mode | Result Return |
|--------|----------------|---------------|
| `Pool.map(func, iterable)` | Synchronous / Blocking | List of results |
| `Pool.map_async(func, iterable)` | Asynchronous / Non-blocking | `AsyncResult` object |
| `Pool.starmap(func, tuple_iterable)` | Synchronous | Unpacks argument tuples |
| `ProcessPoolExecutor.map()` | Concurrent Futures | Iterator of results |


In [ ]:
from concurrent.futures import ProcessPoolExecutor

def compute_square(x):
    return x * x

numbers = [1, 2, 3, 4, 5, 6, 7, 8]

# ProcessPoolExecutor map
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(compute_square, numbers))

print("Input Numbers  :", numbers)
print("Parallel Squares:", results)


---
## 4. Inter-Process Communication (IPC: `Queue` & `Pipe`)

Because processes do not share memory space, communication requires explicitly passed IPC structures:
- **`multiprocessing.Queue`**: Thread- and process-safe FIFO queue.
- **`multiprocessing.Pipe`**: Duplex connection between two processes.


In [ ]:
def producer(q):
    for item in ["data_batch_1", "data_batch_2", "data_batch_3"]:
        q.put(item)
    q.put(None)  # Sentinel value to signal completion

def consumer(q):
    items_processed = []
    while True:
        item = q.get()
        if item is None: break
        items_processed.append(item.upper())
    return items_processed

ipc_queue = mp.Queue()
producer(ipc_queue)
processed = consumer(ipc_queue)
print("IPC Queue Processed Data:", processed)


---
## 5. Shared Memory & Managers (`Value`, `Array`, `Manager`)

- `Value(typecode, arg)` & `Array(typecode, arg)`: Shared C-compatible memory.
- `Manager()`: Server process managing shared Python `dict` and `list` objects.


In [ ]:
# Shared Manager Dictionary
def update_shared_dict(shared_dict, key, value):
    shared_dict[key] = value

with mp.Manager() as manager:
    shared_map = manager.dict()
    
    # Simulated process updates
    update_shared_dict(shared_map, "status", "SUCCESS")
    update_shared_dict(shared_map, "processed_count", 1000)
    
    print("Shared Manager Dict:", dict(shared_map))


---
## 6. Process Synchronization Primitives (`Lock`, `Semaphore`, `Event`)

When multiple processes access a shared resource (file, shared memory, terminal), use a `Lock` to prevent race conditions.


In [ ]:
# Lock usage example
lock = mp.Lock()

def safe_critical_section(lock, process_id):
    with lock:  # Acquires lock, automatically releases on exit
        print(f"  [Process-{process_id}] Entered critical section safely.")

safe_critical_section(lock, 1)
safe_critical_section(lock, 2)


---
## 7. Process Start Methods (`spawn`, `fork`, `forkserver`)

| Method | Platforms | Description |
|--------|-----------|-------------|
| **`spawn`** | Windows, macOS (default), Unix | Starts fresh Python interpreter process. Safest, no resource inheritance leaks. |
| **`fork`** | Unix / Linux | Clones parent process memory. Fast start, but thread-unsafe. |
| **`forkserver`** | Unix | Spawns clean single-threaded server process to handle future forks. |


---
## 8. Empirical Benchmark: Sequential vs. Threads vs. Processes

Demonstrating GIL impact on a CPU-bound task (heavy mathematical computation).


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def cpu_bound_task(n):
    count = 0
    for i in range(n):
        count += i * i
    return count

N = 5_000_000
tasks = [N, N, N, N]

# 1. Sequential Execution
t0 = time.perf_counter()
res_seq = [cpu_bound_task(n) for n in tasks]
t_seq = time.perf_counter() - t0

# 2. Multithreaded Execution (Blocked by GIL)
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    res_thread = list(executor.map(cpu_bound_task, tasks))
t_thread = time.perf_counter() - t0

# 3. Multiprocessing Execution (Bypasses GIL)
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    res_process = list(executor.map(cpu_bound_task, tasks))
t_process = time.perf_counter() - t0

print(f"1. Sequential Time  : {t_seq:.4f}s")
print(f"2. Multithreaded Time: {t_thread:.4f}s (No speedup due to GIL!)")
print(f"3. Multiprocessing Time: {t_process:.4f}s (Speedup: ~{t_seq / t_process:.2f}x!)")


---
## 9. Quick Reference Card


In [ ]:
# ==================================================================
# MULTIPROCESSING – QUICK REFERENCE
# ==================================================================
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

# --- ProcessPoolExecutor ---
# with ProcessPoolExecutor(max_workers=4) as ex:
#     results = list(ex.map(fn, data))

# --- Process Class ---
# p = mp.Process(target=fn, args=(1,))
# p.start(); p.join()


---
## Summary

| Technique / Module | Best Used For | Key Advantage |
|-------------------|---------------|---------------|
| **`ProcessPoolExecutor`** | Batch parallel mapping | Clean context-managed API |
| **`mp.Queue`** | Task passing between processes | Thread & Process safe |
| **`mp.Manager`** | Shared dict / list states | Flexible high-level IPC |
| **`mp.Lock`** | Protecting shared resources | Race condition prevention |

---
*Next up: **Multithreading & Async I/O***
